# 01 - Bronze Layer

Read raw CSV files from the landing Volume and store them as Delta tables.
- No data cleaning
- Add ingestion metadata (_ingestion_timestamp, _source_file)
- Write to `workspace.bronze` schema

In [0]:
VOLUME_PATH = '/Volumes/workspace/retail_landing/raw_data'
BRONZE_SCHEMA = 'workspace.bronze'

print(f'reading from: {VOLUME_PATH}')
print(f"writing to: {BRONZE_SCHEMA}")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df = spark.read \
    .option("header", "true") \
    .option('inferSchema', 'true') \
    .csv(f"{VOLUME_PATH}/olist_customers_dataset.csv")

# Show the first 5 rows
print(f"Total rows: {df.count()}")
print(f"Columns: {df.columns}")
display(df.limit(5))
 

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Add metadata columns
df_bronze = df \
    .withColumn("_ingestion_timestamp", current_timestamp()) \
    .withColumn("_source_file", lit("olist_customers_dataset.csv"))

# Write as a Delta table to the Bronze schema
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.bronze.customers")

print("✅ Bronze table 'customers' created!")
display(spark.table("workspace.bronze.customers").limit(5))

## LOAD TABLES 

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Map each CSV file to a table name
tables = {
    "olist_customers_dataset.csv":         "customers",
    "olist_orders_dataset.csv":            "orders",
    "olist_order_items_dataset.csv":       "order_items",
    "olist_order_payments_dataset.csv":    "order_payments",
    "olist_order_reviews_dataset.csv":     "order_reviews",
    "olist_products_dataset.csv":          "products",
    "olist_sellers_dataset.csv":           "sellers",
    "olist_geolocation_dataset.csv":       "geolocation",
    "product_category_name_translation.csv": "product_category_translation"
}

# Loop through each file and save as a Bronze Delta table
for filename, table_name in tables.items():
    print(f"⏳ Processing {filename}...")
    
    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(f"{VOLUME_PATH}/{filename}") \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source_file", lit(filename))
    
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{BRONZE_SCHEMA}.{table_name}")
    
    print(f"  ✅ {BRONZE_SCHEMA}.{table_name} — {df.count()} rows")

print("\n🎉 All Bronze tables created!")

In [0]:
%sql
SHOW TABLES IN workspace.bronze;

In [0]:
print("Bronze Layer Summary:\n")

for table_name in tables.values():
    count = spark.table(f"{BRONZE_SCHEMA}.{table_name}").count()
    print(f"  {table_name}: {count:,} rows")